# Pandas Assignment
Dataset: `company_data.csv` .

In [1]:
import pandas as pd

## 1. Data Loading & Inspection
Load the `company_data.csv` file. Print the first 10 
rows and use a Pandas method to show the data types and check for missing 
values in each column. 

In [2]:
df = pd.read_csv("company_data.csv")
print("*"*10+" First 10 Rows "+"*"*10)
print(df.head(10))

********** First 10 Rows **********
   Employee_ID Employee_Name Department    Salary  Years_Experience  \
0          101      John Doe         IT   85000.0                 5   
1          102    Jane Smith         HR   60000.0                 3   
2          103   Ali Mansour      Sales       NaN                12   
3          104      Sara Lee         IT   95000.0                 1   
4          105     Ben Clark    Finance   70000.0                 8   
5          106     Eva Green      Sales   55000.0                 4   
6          107      Mona Ray         HR       NaN                 2   
7          108      Leo King         IT  105000.0                10   
8          109      Tom Cook    Finance   72000.0                 6   
9          110     Anna Bell         IT   88000.0                 9   

   Evaluation_Score   Bonus  
0                88  5000.0  
1                72  3000.0  
2                95  7000.0  
3                45     NaN  
4                60  4000.0  
5 

In [3]:
# Data types and non-null counts
print("\n DataFrame Info & Data Types ")
df.info()
print("\n Data Types per Column ")
print(df.dtypes)


 DataFrame Info & Data Types 
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Employee_ID       20 non-null     int64  
 1   Employee_Name     20 non-null     str    
 2   Department        20 non-null     str    
 3   Salary            18 non-null     float64
 4   Years_Experience  20 non-null     int64  
 5   Evaluation_Score  20 non-null     int64  
 6   Bonus             18 non-null     float64
dtypes: float64(2), int64(3), str(2)
memory usage: 1.2 KB

 Data Types per Column 
Employee_ID           int64
Employee_Name           str
Department              str
Salary              float64
Years_Experience      int64
Evaluation_Score      int64
Bonus               float64
dtype: object


In [4]:
# Number of missing values in each column
print("\n  Missing Values per Column \n")
print(df.isna().sum())


  Missing Values per Column 

Employee_ID         0
Employee_Name       0
Department          0
Salary              2
Years_Experience    0
Evaluation_Score    0
Bonus               2
dtype: int64


## 2. Handling Missing Values
The dataset contains missing values in the `Salary` and `Bonus` columns.
- Fill the missing `Salary` values with the median salary of the entire company.
- Fill the missing `Bonus` values with 0.

In [5]:
median_salary = df["Salary"].median()
df["Salary"] = df["Salary"].fillna(median_salary)
df["Bonus"] = df["Bonus"].fillna(0)
print(df[["Salary", "Bonus"]].isnull().sum())

Salary    0
Bonus     0
dtype: int64


## 3. Data Cleaning
The `Employee_Name` column is in the format "FirstName LastName". Split this column into two new columns: `First_Name` and `Last_Name`, then delete the original `Employee_Name` column.

In [6]:
df[["First_Name", "Last_Name"]] = df["Employee_Name"].str.strip().str.split(n=1, expand=True)
df = df.drop(columns="Employee_Name")
print(df[["First_Name", "Last_Name"]].head())
print("Employee_Name exists:", "Employee_Name" in df.columns)

  First_Name Last_Name
0       John       Doe
1       Jane     Smith
2        Ali   Mansour
3       Sara       Lee
4        Ben     Clark
Employee_Name exists: False


## 4. Filtering
Create a new DataFrame called `Senior_IT` that contains only employees from the 'IT' department who have more than 8 years of experience.

In [7]:
# Find the experience column (its name contains "exp")
exp_col = [col for col in df.columns if "exp" in col.lower()][0]

Senior_IT = df[(df["Department"] == "IT") & (df[exp_col] > 8)].copy()
print("Number of Senior IT employees:", len(Senior_IT))
print(Senior_IT.head())

Number of Senior IT employees: 5
    Employee_ID Department    Salary  Years_Experience  Evaluation_Score  \
7           108         IT  105000.0                10                91   
9           110         IT   88000.0                 9                40   
12          113         IT   92000.0                11                77   
15          116         IT  110000.0                15                98   
18          119         IT   98000.0                 9                81   

     Bonus First_Name Last_Name  
7   8000.0        Leo      King  
9   2500.0       Anna      Bell  
12  6000.0       Will     Smith  
15  9000.0      Bruce     Wayne  
18  6500.0      Barry     Allen  


## 5. Performance Mapping
Create a new column called `Performance_Level` based on the `Evaluation_Score` (1-100):
- Score > 80: "High"
- Score 50-80: "Medium"
- Score < 50: "Low"

In [8]:
def performance_level(score):
    if pd.isna(score):
        return None      # no score, so no level
    elif score > 80:
        return "High"
    elif score >= 50:
        return "Medium"
    else:
        return "Low"

df["Performance_Level"] = df["Evaluation_Score"].apply(performance_level)
df[["Evaluation_Score", "Performance_Level"]].head(10)

,Evaluation_Score,Performance_Level
0,88,High
1,72,Medium
2,95,High
3,45,Low
4,60,Medium
5,82,High
6,55,Medium
7,91,High
8,78,Medium
9,40,Low


## 6. Department Summary
Calculate the total salary expense and the average evaluation score for each Department.

In [9]:
department_summary = df.groupby("Department").agg(
    Total_Salary=("Salary", "sum"),
    Average_Score=("Evaluation_Score", "mean")
)
department_summary["Average_Score"] = department_summary["Average_Score"].round(2)
print(department_summary)

            Total_Salary  Average_Score
Department                             
Finance         294000.0          74.25
HR              258500.0          71.50
IT              673000.0          74.29
Sales           299500.0          67.60


## 7. Top 5 Highest-Paid Employees
Find the top 5 highest-paid employees in the company. Print only their names and salaries.

In [10]:
top_5 = df.nlargest(5, "Salary")[["First_Name", "Last_Name", "Salary"]]
print(top_5.to_string(index=False))

First_Name Last_Name   Salary
     Bruce     Wayne 110000.0
       Leo      King 105000.0
     Barry     Allen  98000.0
      Sara       Lee  95000.0
      Will     Smith  92000.0


## 8. Sales Raise
The company is giving a 10% raise to everyone in the 'Sales' department. Update 
the `Salary` column for those employees only. 

In [11]:
# Convert to float because the new salaries 
df["Salary"] = df["Salary"].astype(float)
sales = df["Department"] == "Sales"
df.loc[sales, "Salary"] = df.loc[sales, "Salary"] * 1.10
print(df[df["Department"] == "Sales"][["First_Name", "Last_Name", "Department", "Salary"]].head(10))

   First_Name Last_Name Department   Salary
2         Ali   Mansour      Sales  80850.0
5         Eva     Green      Sales  60500.0
10       Ryan       Gos      Sales  63800.0
13       Kate      Wins      Sales  59400.0
17      Diana    Prince      Sales  64900.0


## 9. Exporting Results
Save the final, cleaned, and updated DataFrame to a new Excel or CSV file named `final_employee_report.csv`.

In [12]:
df.to_csv("final_employee_report.csv", index=False)
